In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [2]:
file   = "/home/ordn/Documents/ordn_projects/Gitmore-Transformer/commits.txt"
data   = open(file, "r", encoding='utf-16').read().splitlines()[:70000]
print(f"the total number of commits = {len(data)}")

the total number of commits = 70000


In [3]:
vocabs = ['^'] + list(sorted(set("".join(data)))) + ['~']

In [4]:
print(f"{len(vocabs)} vocabs = {'|'.join(vocabs)}")

95 vocabs = ^| |!|"|#|$|%|&|'|(|)|*|+|,|-|.|/|0|1|2|3|4|5|6|7|8|9|:|;|<|=|>|?|@|A|B|C|D|E|F|G|H|I|J|K|L|M|N|O|P|Q|R|S|T|U|V|W|X|Y|Z|[|\|]|_|`|a|b|c|d|e|f|g|h|i|j|k|l|m|n|o|p|q|r|s|t|u|v|w|x|y|z|{|||}|~


In [5]:
stoi   = {s:i for i,s in enumerate(vocabs)}
itos   = {i:s for i,s in enumerate(vocabs)}

In [6]:
traindata, valdata = data[:int(len(data)*0.8)], data[int(len(data)*0.8):]

In [7]:
def encode(data: list):

  data = data if isinstance(data, list) else [data]
  ix   = []
  for line in data:
    ix.append([stoi[s] for s in list('^' + line + '~')])
  x    = []
  for dx in ix:
    x += dx
  return x

def decode(x):
  out  = []
  for ix in x:
    out.append(itos[ix])
  return ''.join(out)

In [8]:
enctrain = encode(traindata)
encval   = encode(valdata)

In [9]:
def get_data(state='train'):
  encoded_data = enctrain if state == 'train' else encval
  inx   = torch.randint(0, (len(encoded_data)-(block_size+1)), (1,)).item()
  x     = torch.tensor(encoded_data[inx:inx+block_size], dtype=torch.long)
  y     = torch.tensor(encoded_data[inx+1:(inx+block_size)+1], dtype=torch.long)
  return x, y

In [10]:
# model hyperParameters
block_size = 32
vocab_size = len(vocabs)
n_embed    = 128 
n_hidden   = 100
num_heads  = 4

In [11]:
class LearnedPE(nn.Module):
  def __init__(self, max_seq_len:int, n_embed:int):
    super().__init__()
    self.emb = nn.Embedding(max_seq_len, n_embed)

  def forward(self,x):
    seq_len = x.shape[0]
    inn     = torch.arange(0, seq_len)
    out     = self.emb(inn)
    return x + out

In [12]:
class ResidualBlock(nn.Module):
  def __init__(self, P, just_capture=False):
    super().__init__()
    self.just_capture = just_capture
    self.P = P

  def forward(self, x):
    if self.just_capture:
      self.P.x = x
      return x

    out = x + self.P.x
    self.P.x = out
    return out

In [13]:
class CasualAttentionModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.model  = nn.Sequential(
      nn.Embedding(vocab_size, n_embed),         # 0 
      LearnedPE(150, n_embed),                   # 1
      ResidualBlock(self,just_capture=True),     # 2
      nn.MultiheadAttention(n_embed, num_heads), # 3
      nn.Linear(n_embed, n_embed, bias=False),   # 4
      nn.LayerNorm(n_embed),                     # 5
      ResidualBlock(self, False),                # 6
      nn.Tanh(),                                 # 7
      nn.MultiheadAttention(n_embed, num_heads), # 8
      ResidualBlock(self, False),                # 9
      nn.Linear(n_embed, n_hidden, bias=False),  # 10
      nn.LayerNorm(n_hidden),                    # 11
      nn.Tanh(),                                 # 12
      nn.Linear(n_hidden, vocab_size, bias=True),# 13
      )
    print(f"total number of parameters = {sum([p.nelement() for p in self.parameters()])}")

  def forward(self, x, y=None):
    x  = self.model[:3](x)
    T  = x.shape[0]
    mask   = torch.triu(torch.ones(T, T), 1).bool() # masking
    x,_= self.model[3](x,x,x, attn_mask=mask)
    x  = self.model[4:8](x) # first Tanh
    x,_= self.model[8](x,x,x, attn_mask=mask)
    logits  = self.model[9:](x)
    if y is not None: # (T)
      loss = F.cross_entropy(logits, y)
    else:
      loss = None
    return logits, loss

  def generate(self):
    out = []
    inn = torch.tensor([stoi['^']])
    while True:
      logits, _ = self(inn)
      probs     = F.softmax(logits[-1], dim=0) if logits.ndim > 1 else F.softmax(logits, dim=0)
      ix        = torch.multinomial(probs, num_samples=1).item()
      if ix != stoi['~']:
        out.append(itos[ix])
        lin = inn.view(-1).tolist(); lin.append(ix)
        if len(lin) > block_size:
          lin = lin[1:]
        inn = torch.tensor(lin)
      else:
        break
    print(''.join(out))

  def fit(self, epochs=1000, lr=1e-3):

    self.optimizer = optim.AdamW(self.parameters(), lr=lr)
    for i in range(epochs):

      self.optimizer.zero_grad(set_to_none=True)
      x, y   = get_data('train')
      xv, yv = get_data('val')

      _,loss= self(x, y)
      loss.backward()

      # update
      self.optimizer.step()

      # validation
      with torch.no_grad():
        _, valloss = self(xv, yv)

      if (i+1) % max(1, int(epochs/10)) == 0:
        print(f"epoch:{i+1}   | loss={loss.item():.4f}   |  val loss={valloss.item():.4f}")

In [14]:
model = CasualAttentionModel()

total number of parameters = 202691


In [26]:
model.fit(epochs=50000)

epoch:5000   | loss=1.4073   |  val loss=1.8023
epoch:10000   | loss=2.0526   |  val loss=1.3552
epoch:15000   | loss=1.5761   |  val loss=2.8603
epoch:20000   | loss=1.5239   |  val loss=1.1854
epoch:25000   | loss=1.0465   |  val loss=1.7085
epoch:30000   | loss=2.4503   |  val loss=1.5642
epoch:35000   | loss=1.3764   |  val loss=1.5129
epoch:40000   | loss=1.3473   |  val loss=1.7776
epoch:45000   | loss=1.6546   |  val loss=1.2902
epoch:50000   | loss=1.8872   |  val loss=2.3456


In [65]:
model.generate()

bpo-300413: Fix PySHavi6) entry ofined terfile_feont-inssnect confie shed of fweents of outdated mutstred.
